**Double Check A-Star Dataset**

In [12]:
# =========================
# Inspect A* dataset by BACK-count category
# =========================

import re
import random
from collections import defaultdict, Counter

import torch
import pandas as pd
from IPython.display import display, Markdown



# -------------------------
# Macros / config
# -------------------------
DATASET_PATH = "/workspace/user/chainsearch_archives/data_pts/astar/astar_data_NALL_S6-6_T{6:100-999}_exp2500_trace10000_sw8_stick0_compact0_useh20_h2all0_h2k1.0_seed42_fmtv2.pt"
SAMPLES_PER_CATEGORY = 3
MAX_BACKTRACKS = 8
SEED = 42

# If False, long traces get truncated in the printout
SHOW_FULL_TRACE = True
TRACE_CHAR_LIMIT = 1500

# -------------------------
# Load dataset
# -------------------------
data = torch.load(DATASET_PATH, map_location="cpu")

prompts = data.get("prompts") or data.get("ins")
traces  = data.get("traces")  or data.get("outs")

if prompts is None or traces is None:
    raise ValueError("Could not find prompts/traces in dataset file.")

if len(prompts) != len(traces):
    raise ValueError(f"Length mismatch: len(prompts)={len(prompts)}, len(traces)={len(traces)}")

print(f"Loaded dataset: {DATASET_PATH}")
print(f"Num samples: {len(prompts):,}")

# -------------------------
# Helpers
# -------------------------
RE_BACK = re.compile(r"\bBACK\b")
RE_STEP = re.compile(r"(?:\[\s*[^\]]*?\s*\]\s*->\s*)?\d+\s*[+\-*/]\s*\d+\s*=\s*\d+\s*->\s*\[\s*[^\]]*?\s*\]\s*;")

def count_backs(trace: str) -> int:
    return len(RE_BACK.findall(trace))

def count_steps(trace: str) -> int:
    return len(RE_STEP.findall(trace))

def shorten(text: str, limit: int = TRACE_CHAR_LIMIT) -> str:
    if SHOW_FULL_TRACE or len(text) <= limit:
        return text
    return text[:limit] + "\n... [truncated] ..."

def format_trace_only(trace: str) -> str:
    s = trace.strip()
    s = s.replace("; ", ";\n")
    s = s.replace("SUCCESS ;\nEXPR", "SUCCESS ; EXPR")
    s = re.sub(r"\n{3,}", "\n\n", s).strip()
    return s

# -------------------------
# Build per-sample metadata
# -------------------------
rows = []
for i, (p, t) in enumerate(zip(prompts, traces)):
    b = count_backs(t)
    s = count_steps(t)
    rows.append({
        "idx": i,
        "backtracks": b,
        "steps": s,
        "prompt": p,
        "trace": t,
    })

df = pd.DataFrame(rows)

# -------------------------
# Summary table
# -------------------------
summary_rows = []
for b in range(MAX_BACKTRACKS + 1):
    sub = df[df["backtracks"] == b]
    summary_rows.append({
        "backtracks": b,
        "count": len(sub),
        "% of dataset": 100.0 * len(sub) / len(df),
        "mean_steps": sub["steps"].mean() if len(sub) else float("nan"),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Optional: also show any out-of-range categories if they exist
extra = df[df["backtracks"] > MAX_BACKTRACKS]
if len(extra):
    print(f"Warning: found {len(extra):,} samples with backtracks > {MAX_BACKTRACKS}")
    print(extra["backtracks"].value_counts().sort_index())

# -------------------------
# Random sampling by category
# -------------------------
rng = random.Random(SEED)

for b in range(MAX_BACKTRACKS + 1):
    sub = df[df["backtracks"] == b]
    n_avail = len(sub)
    n_take = min(SAMPLES_PER_CATEGORY, n_avail)

    display(Markdown(f"## BACK count = {b}  \nAvailable: **{n_avail:,}** | Sampling: **{n_take}**"))

    if n_take == 0:
        print("No samples in this category.\n")
        continue

    chosen_idxs = rng.sample(list(sub.index), n_take)

    for j, ridx in enumerate(chosen_idxs, start=1):
        row = df.loc[ridx]

        print("=" * 100)
        print(f"Category BACK={b} | Sample {j}/{n_take} | dataset idx={row['idx']} | steps={row['steps']}")
        print("- PROMPT:")
        print(row["prompt"])
        print("- TRACE:")
        print(shorten(format_trace_only(row["trace"])))
        print()

# -------------------------
# Optional: raw value counts
# -------------------------
print("\nRaw BACK-count distribution:")
print(df["backtracks"].value_counts().sort_index())

/tmp/ipykernel_13188/629713910.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(DATASET_PATH, map_location="cpu")


Loaded dataset: /workspace/user/chainsearch_archives/data_pts/astar/astar_data_NALL_S6-6_T{6:100-999}_exp2500_trace10000_sw8_stick0_compact0_useh20_h2all0_h2k1.0_seed42_fmtv2.pt
Num samples: 1,016,676


,backtracks,count,% of dataset,mean_steps
0,0,219827,21.622129,1.737562
1,1,20066,1.973687,3.209160
2,2,106211,10.446888,4.198529
3,3,126931,12.484902,5.398925
4,4,120193,11.822154,6.741907
5,5,109666,10.786721,8.165995
6,6,106491,10.474428,9.603788
7,7,106566,10.481805,11.052005
8,8,100725,9.907286,12.580541


## BACK count = 0  
Available: **219,827** | Sampling: **3**

Category BACK=0 | Sample 1/3 | dataset idx=775475 | steps=2
- PROMPT:
IN: [ 5, 7, 7, 9, 25, 100 ] TGT: 893
- TRACE:
[ 5, 7, 7, 9, 25, 100 ] -> 9 * 100 = 900 -> [ 5, 7, 7, 25, 900 ] ;
[ 5, 7, 7, 25, 900 ] -> 900 - 7 = 893 -> [ 5, 7, 25, 893 ] ;
SUCCESS ; EXPR (((9 * 100) - 7))

Category BACK=0 | Sample 2/3 | dataset idx=133826 | steps=2
- PROMPT:
IN: [ 2, 3, 5, 5, 7, 25 ] TGT: 178
- TRACE:
[ 2, 3, 5, 5, 7, 25 ] -> 7 * 25 = 175 -> [ 2, 3, 5, 5, 175 ] ;
[ 2, 3, 5, 5, 175 ] -> 3 + 175 = 178 -> [ 2, 5, 5, 178 ] ;
SUCCESS ; EXPR ((3 + (7 * 25)))

Category BACK=0 | Sample 3/3 | dataset idx=30545 | steps=1
- PROMPT:
IN: [ 2, 6, 7, 9, 75, 100 ] TGT: 675
- TRACE:
[ 2, 6, 7, 9, 75, 100 ] -> 9 * 75 = 675 -> [ 2, 6, 7, 100, 675 ] ;
SUCCESS ; EXPR ((9 * 75))



...

Category BACK=6 | Sample 1/3 | dataset idx=114126 | steps=9
- PROMPT:
IN: [ 3, 4, 6, 8, 50, 100 ] TGT: 211
- TRACE:
[ 3, 4, 6, 8, 50, 100 ] -> 4 * 50 = 200 -> [ 3, 6, 8, 100, 200 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 3 * 50 = 150 -> [ 4, 6, 8, 100, 150 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 50 + 100 = 150 -> [ 3, 4, 6, 8, 150 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 3 * 100 = 300 -> [ 4, 6, 8, 50, 300 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 6 * 50 = 300 -> [ 3, 4, 8, 100, 300 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 8 + 100 = 108 -> [ 3, 4, 6, 50, 108 ] ;
BACK -> [ 3, 4, 6, 8, 50, 100 ] ;
[ 3, 4, 6, 8, 50, 100 ] -> 4 * 50 = 200 -> [ 3, 6, 8, 100, 200 ] ;
[ 3, 6, 8, 100, 200 ] -> 8 + 200 = 208 -> [ 3, 6, 100, 208 ] ;
[ 3, 6, 100, 208 ] -> 3 + 208 = 211 -> [ 6, 100, 211 ] ;
SUCCESS ; EXPR ((3 + (8 + (4 * 50))))

Category BACK=6 | Sample 2/3 | dataset idx=273554 | 

## BACK count = 7  
Available: **106,566** | Sampling: **3**

Category BACK=7 | Sample 1/3 | dataset idx=632194 | steps=11
- PROMPT:
IN: [ 5, 9, 9, 10, 75, 100 ] TGT: 435
- TRACE:
[ 5, 9, 9, 10, 75, 100 ] -> 9 * 75 = 675 -> [ 5, 9, 10, 100, 675 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 5 * 75 = 375 -> [ 9, 9, 10, 100, 375 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 5 * 100 = 500 -> [ 9, 9, 10, 75, 500 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 10 * 75 = 750 -> [ 5, 9, 9, 100, 750 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 10 + 100 = 110 -> [ 5, 9, 9, 75, 110 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 5 + 100 = 105 -> [ 9, 9, 10, 75, 105 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 10 * 75 = 750 -> [ 5, 9, 9, 100, 750 ] ;
[ 5, 9, 9, 100, 750 ] -> 750 - 100 = 650 -> [ 5, 9, 9, 650 ] ;
BACK -> [ 5, 9, 9, 10, 75, 100 ] ;
[ 5, 9, 9, 10, 75, 100 ] -> 5 * 100 = 500 -> [ 9, 9, 10, 75, 500 ] ;
[ 9, 9, 10, 75, 500 ] 

## BACK count = 8  
Available: **100,725** | Sampling: **3**

Category BACK=8 | Sample 1/3 | dataset idx=739463 | steps=14
- PROMPT:
IN: [ 1, 2, 4, 4, 7, 75 ] TGT: 144
- TRACE:
[ 1, 2, 4, 4, 7, 75 ] -> 2 * 75 = 150 -> [ 1, 4, 4, 7, 150 ] ;
[ 1, 4, 4, 7, 150 ] -> 150 - 7 = 143 -> [ 1, 4, 4, 143 ] ;
BACK -> [ 1, 4, 4, 7, 150 ] ;
[ 1, 4, 4, 7, 150 ] -> 150 - 4 = 146 -> [ 1, 4, 7, 146 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 7 + 75 = 82 -> [ 1, 2, 4, 4, 82 ] ;
[ 1, 2, 4, 4, 82 ] -> 2 * 82 = 164 -> [ 1, 4, 4, 164 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 4 + 75 = 79 -> [ 1, 2, 4, 7, 79 ] ;
[ 1, 2, 4, 7, 79 ] -> 2 * 79 = 158 -> [ 1, 4, 7, 158 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 7 - 1 = 6 -> [ 2, 4, 4, 6, 75 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 2 - 1 = 1 -> [ 1, 4, 4, 7, 75 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 2 + 4 = 6 -> [ 1, 4, 6, 7, 75 ] ;
BACK -> [ 1, 2, 4, 4, 7, 75 ] ;
[ 1, 2, 4, 4, 7, 75 ] -> 4 / 4 = 1 -> [ 1, 1, 2, 7, 75 ] ;
BACK -> [ 